# Data Merging — Stage 4 Assembly 01: Apply Union

## Input
- `Data/Data_Collection/Final/Stage_4_Normalised/{5 aggregate table names}.parquet` (from Stage 3 notebooks 04/05)
- `Data/Data_Collection/Final/Stage_4_Normalised_Panel/{2 panel table names}.parquet`
- `Data/Data_Collection/Final/Stage_4_Normalised/union_drop_list.csv`
- `Data/Data_Collection/Final/Stage_4_Normalised/decisions_aggregate.csv`
- `Data/Data_Collection/Final/Stage_4_Normalised_Panel/decisions_panel.csv`
- `lib.review` (`base_factor_map`) and `lib.config` (`AGG_Z`, `PANEL_Z`, `OUT`, `META`, `BINARIES`)

## Purpose
The first notebook in Stage 4 Assembly. Takes the seven z-scored tables produced by Stage 3 Cleaning and applies every accumulated drop decision to them in one place, producing a set of tables where all seven agree on the same base factor set at the stock level. This is where the union drop list, the per-table rule drops, and a small set of manual/structural corrections all converge into the final assembled feature tables.

## Three Drop Sources
1. **Union** (`union_drop_list.csv`), keyed on **base factor**. Contains a base factor only when its cap-weighted mean failed, so applying it removes the whole factor from both the aggregate and panel pipelines.
2. **Rule drops** (`decisions_aggregate.csv` / `decisions_panel.csv`), keyed on **column**. The union alone is *not* sufficient for the aggregate side: 28 base factors kept a surviving `cwmean` but lost an individual higher moment (`_cwstd`, `_cwskew`, `_cwkurt`, `_spread`) to a specific rule (e.g. `Tax_cwkurt`, `RoE_cwstd`, `AOP_cwstd`). Those moment-only drops live only in `decisions_aggregate.csv` and aren't captured by the base-factor-level union. The panel has no moments, so every panel-side drop is already a whole base factor and is already a subset of the union.
3. **Manual** (`MANUAL_DROP`), a small hardcoded set of individual exceptions:
   - `stock_skew_chg_5d` — exists in `agg_market_daily_means` with no counterpart in the full-moments table, a Stage 2 construction asymmetry not worth chasing given how many other factors already exist.
   - Twelve stock-level factors (`CBOperProf`, `NetPayoutYield`, `AOP`, `GrSaleToGrOverhead`, `GP`, `MomSeason16YrPlus`, `GrSaleToGrInv`, `MomOffSeason16YrPlus`, `DivYieldST`, `fgr5yrLag`, `MomSeason11YrPlus`, `InvestPPEInv`) — all above 10% imputed in the assembled panel (measured separately in notebook 03's `fill_report_panel.csv`), predominantly long-horizon momentum and accounting factors unavailable for recently-added index entrants. Together these carry 898,506 missing cells: 22% of *all* panel imputation coming from just 4% of features.

   **Why dropped rather than tolerated:** zero-filling a z-score places every imputed row at exactly `x = 0`, and B-spline basis functions there are non-zero constants — so every imputed row pushes an identical gradient direction into the coefficients near the origin. With `grid_size=14` over `[-5.5, 5.5]`, the central bin already holds ~31% of a normal feature's mass; at 23.7% imputation, that bin becomes roughly half fabricated. Worse, the missingness is non-random — it rises from 1.8% in 2007 to 3.8% in 2024 — so the imputation systematically pulls the fit toward the behaviour of thinner-coverage (i.e. newer, smaller) stocks rather than cancelling out as noise would.

`rule_drops()` prefers the `final_action` column (post-union outcome, written by the panel notebook) over the raw `action` column where available, so this notebook is reading each pipeline's *final* verdict rather than its pre-union one.

## Pass 1 — Union, Rule, and Manual Drops
For each of the seven tables:
- Computes `bmap = rv.base_factor_map(df.columns)` using the table's **full column list**, not a bare `base_factor()` call per column — this is called out explicitly because the `_spread` suffix only correctly strips to its base name when the matching `_cwmean` sibling is visible in the same column set. Without the full list, a bare `X_spread` maps to itself, fails to match anything in the union list, and survives orphaned while its four sibling moments are removed.
- Computes three disjoint column sets: `by_union` (base factor in the union list), `by_rule` (column-level drop from the decisions file, minus anything already caught by union), `by_manual` (base factor in `MANUAL_DROP`, minus anything already caught by union or rule).
- Drops the union of all three sets and stores the resulting frame.
- Prints per-table counts for union/rule/manual drops and features kept.

## Pass 2 — Enforce an Identical Stock-Level Base Factor Set
The union drop list only equalizes what's *removed*; it cannot force a column into a pipeline that never had it in the first place. This surfaces as 17 stock factors existing in only one of the two pipelines — 3 ISO columns and 4 PriceDelay/OptionVolume2 factors present only in the panel, and 9 corporate-event/analyst factors present only in the aggregate.

Since the entire point of maintaining parallel panel and aggregate pipelines is to compare spline fits across them, this asymmetry is unacceptable and is explicitly closed here: for each frequency (daily/monthly), computes the stock-level base factor set in both the aggregate full-moments table (via `_cwmean` suffix stripping) and the panel table, takes the symmetric difference, and drops every asymmetric factor from **both** pipelines wherever it currently exists (`extra_drop`).

## Save
Writes all seven finalized tables to `Stage_5_Model_Ready/01_unioned/`, asserting no `META` columns were accidentally lost in the process. Produces `union_applied_report.csv` summarizing, per table: counts dropped by union / rule / manual / identity-enforcement, final feature count, meta column count, and row count. Prints total feature counts summed separately across the aggregate-side tables and the panel-side tables.

## Validation Cells (Post-Save)

**Check 1 — Do panel and aggregate hold the same stock-level factors?**
Re-derives the stock-level base factor set from the *saved* output tables (not the in-memory frames) for both daily and monthly, confirming the asymmetry from Pass 2 was actually resolved. Explicitly frames any remaining asymmetry as "a starting-set difference, not a union failure" — i.e., a check on Pass 2's own correctness, not a re-run of Pass 2's logic.

**Check 2 — Do the means table and the full-moments `_cwmean` column agree?**
For both daily and monthly, verifies that the set of columns in `agg_market_*_means.parquet` exactly equals the union of (stock factors implied by `_cwmean` columns in the full-moments table) + (shared macro columns) + (meta columns) — i.e., that the two aggregate table variants didn't drift apart during the drop process and still describe the same underlying factor set.

**Check 3 — Date coverage**
Reports min/max date and unique date count for every one of the seven saved tables, explicitly noted as informing the `START_DATE` trim decision in notebook 02. Separately checks for dates present in the panel but absent from the aggregate, reporting the count and range of any such dates and specifically how many fall after 2007-08-01 (presumably a relevant cutoff for a later trimming decision).

## Output
- `Data/Data_Collection/Final/Stage_5_Model_Ready/01_unioned/{7 table names}.parquet` — all seven tables with an identical, reconciled stock-level base factor set.
- `Data/Data_Collection/Final/Stage_5_Model_Ready/01_unioned/union_applied_report.csv` — per-table breakdown of drop counts by source (union / rule / manual / identity) and final shape.

In [1]:
import sys
from pathlib import Path
import pandas as pd

sys.path.append('../..')
import lib.review as rv
from lib.config import AGG_Z, PANEL_Z, OUT, META, BINARIES

OUT_DIR = OUT / '01_unioned'
OUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCES = {
    'agg_market_daily_means':          AGG_Z,
    'agg_market_daily_full_moments':   AGG_Z,
    'weekly_raw':                      AGG_Z,
    'agg_market_monthly_means':        AGG_Z,
    'agg_market_monthly_full_moments': AGG_Z,
    'panel_stock_daily_engineered':    PANEL_Z,
    'panel_stock_monthly_engineered':  PANEL_Z,
}

# ── THREE DROP SOURCES ───────────────────────────────────────────────────────
#
# 1. UNION, keyed on BASE factor. Contains a base factor only when its
#    cap-weighted mean failed, so it removes whole factors from both pipelines.
#
# 2. RULE DROPS, keyed on COLUMN. The union list is NOT sufficient for the
#    aggregate: 28 base factors kept their cwmean but lost individual moments
#    to R2 or R6 (Tax_cwkurt, RoE_cwstd, AOP_cwstd...). Those live only in
#    decisions_aggregate.csv. The panel has no moments, so every panel drop is
#    a whole base factor and its rule drops are already a subset of the union.
#
# 3. MANUAL, one column. stock_skew_chg_5d exists in agg_market_daily_means but
#    has no counterpart in the full-moments table -- a Stage 2 asymmetry. Not
#    worth chasing with ~300 factors already.

union_drop = set(pd.read_csv(AGG_Z / 'union_drop_list.csv')['base_factor'])
MANUAL_DROP = {'stock_skew_chg_5d'}

# Stock-level features above 10% imputed in the assembled panel (measured in
# notebook 03, fill_report_panel.csv). Predominantly long-horizon momentum and
# accounting factors unavailable for recent index entrants. Together they carry
# 898,506 missing cells -- 22% of all panel imputation from 4% of the features.
#
# Dropped rather than tolerated because zero-filling a z-score places the
# imputed rows at exactly x=0, and B-spline basis functions there are non-zero
# constants: every imputed row sends an identical gradient direction to the
# coefficients near the origin. With grid_size=14 over [-5.5, 5.5] the central
# bin holds ~31% of a normal feature's mass, so at 23.7% imputation that bin is
# roughly half fabricated. Missingness is also non-random -- it rises from 1.8%
# in 2007 to 3.8% in 2024 -- so the pull is toward the behaviour of
# thinner-coverage stocks rather than cancelling out.
MANUAL_DROP |= {
    'CBOperProf', 'NetPayoutYield', 'AOP', 'GrSaleToGrOverhead', 'GP',
    'MomSeason16YrPlus', 'GrSaleToGrInv', 'MomOffSeason16YrPlus',
    'DivYieldST', 'fgr5yrLag', 'MomSeason11YrPlus', 'InvestPPEInv',
}

DEC_PATH = {t: (AGG_Z / 'decisions_aggregate.csv' if src is AGG_Z
                else PANEL_Z / 'decisions_panel.csv')
            for t, src in SOURCES.items()}
_dec_cache = {p: pd.read_csv(p) for p in set(DEC_PATH.values())}

def rule_drops(tag):
    """Column-level drops for one table. Prefers final_action (post-union)
    where the panel notebook wrote it."""
    d = _dec_cache[DEC_PATH[tag]]
    d = d[d['table_source'] == tag]
    col = 'final_action' if 'final_action' in d.columns else 'action'
    return set(d.loc[d[col] == 'drop', 'feature'])

print(f'union_drop_list.csv : {len(union_drop)} base factors')
print(f'manual              : {sorted(MANUAL_DROP)}')

# ── PASS 1: union + rule + manual ────────────────────────────────────────────

print('\n' + '=' * 96)
print('PASS 1 - union, rule and manual drops')
print('=' * 96)

frames, rows = {}, []
for tag, src in SOURCES.items():
    df = pd.read_parquet(src / f'{tag}.parquet')
    meta = [c for c in META[tag] + BINARIES if c in df.columns]

    # base_factor_map, NOT base_factor per column. The _spread branch only
    # strips when the matching _cwmean is visible, so without the full column
    # list X_spread maps to itself, is not found in the union list, and
    # survives while its four siblings are removed.
    bmap = rv.base_factor_map(df.columns)
    rdrop = rule_drops(tag)

    by_union  = {c for c in df.columns if c not in meta and bmap[c] in union_drop}
    by_rule   = {c for c in df.columns if c not in meta and c in rdrop} - by_union
    by_manual = {c for c in df.columns if c not in meta
                 and bmap[c] in MANUAL_DROP} - by_union - by_rule

    df = df.drop(columns=sorted(by_union | by_rule | by_manual))
    frames[tag] = df

    rows.append({'table': tag, 'by_union': len(by_union), 'by_rule': len(by_rule),
                 'by_manual': len(by_manual), 'features_kept': len(df.columns) - len(meta)})
    print(f'  {tag:<34} union {len(by_union):>4}  rule {len(by_rule):>3}  '
          f'manual {len(by_manual):>2}   kept {len(df.columns) - len(meta):>4}')

# ── PASS 2: enforce an identical stock-level base factor set ─────────────────
#
# The union equalises the DROP lists, but it cannot remove a panel column the
# aggregate never held. 17 stock factors exist in one pipeline only -- 3 ISO
# columns and 4 PriceDelay/OptionVolume2 in the panel, 9 corporate-event and
# analyst factors in the aggregate. Since the spline comparison across the two
# pipelines is the point of the exercise, the intersection is enforced.

def stock_base(tag, df):
    if tag.endswith('full_moments'):
        return {c[:-len('_cwmean')] for c in df.columns if c.endswith('_cwmean')}
    meta = set(META[tag] + BINARIES)
    return {c for c in df.columns if c not in meta}

print('\n' + '=' * 96)
print('PASS 2 - enforce identical stock-level base factors')
print('=' * 96)

extra_drop = set()
for freq, agg_tag, pan_tag in [
    ('DAILY',   'agg_market_daily_full_moments',   'panel_stock_daily_engineered'),
    ('MONTHLY', 'agg_market_monthly_full_moments', 'panel_stock_monthly_engineered'),
]:
    a = stock_base(agg_tag, frames[agg_tag])
    p = stock_base(pan_tag, frames[pan_tag])
    asym = (a - p) | (p - a)
    extra_drop |= asym
    print(f'  {freq:<8} aggregate {len(a):>4}  panel {len(p):>4}  '
          f'shared {len(a & p):>4}  removing {len(asym)}')
    if p - a:
        print(f'    panel only     : {sorted(p - a)}')
    if a - p:
        print(f'    aggregate only : {sorted(a - p)}')

for tag, df in frames.items():
    bmap = rv.base_factor_map(df.columns)
    meta = [c for c in META[tag] + BINARIES if c in df.columns]
    drop = [c for c in df.columns if c not in meta and bmap[c] in extra_drop]
    if drop:
        frames[tag] = df.drop(columns=drop)
    r = next(x for x in rows if x['table'] == tag)
    r['by_identity'] = len(drop)
    r['features_final'] = len(frames[tag].columns) - len(meta)

# ── SAVE ─────────────────────────────────────────────────────────────────────

print('\n' + '=' * 96)
print('SAVE')
print('=' * 96)

for tag, df in frames.items():
    meta = [c for c in META[tag] + BINARIES if c in df.columns]
    missing = [c for c in META[tag] if c not in df.columns]
    assert not missing, f'{tag}: META column(s) lost: {missing}'
    df.to_parquet(OUT_DIR / f'{tag}.parquet', index=False)
    r = next(x for x in rows if x['table'] == tag)
    r['rows'], r['meta'] = len(df), len(meta)
    print(f'  {tag:<34} {r["features_final"]:>4} features   {len(df):>7,} rows')

report = pd.DataFrame(rows)[['table', 'by_union', 'by_rule', 'by_manual',
                             'by_identity', 'features_final', 'meta', 'rows']]
report.to_csv(OUT_DIR / 'union_applied_report.csv', index=False)

agg = report[report['table'].str.startswith('agg') | (report['table'] == 'weekly_raw')]
pan = report[report['table'].str.startswith('panel')]
print(f'\n  aggregate : {agg["features_final"].sum():,} features')
print(f'  panel     : {pan["features_final"].sum():,} features')
print(f'\n  saved -> {OUT_DIR}')

union_drop_list.csv : 85 base factors
manual              : ['AOP', 'CBOperProf', 'DivYieldST', 'GP', 'GrSaleToGrInv', 'GrSaleToGrOverhead', 'InvestPPEInv', 'MomOffSeason16YrPlus', 'MomSeason11YrPlus', 'MomSeason16YrPlus', 'NetPayoutYield', 'fgr5yrLag', 'stock_skew_chg_5d']

PASS 1 - union, rule and manual drops
  agg_market_daily_means             union   40  rule   0  manual  1   kept  286
  agg_market_daily_full_moments      union  168  rule   5  manual  0   kept  818
  weekly_raw                         union    2  rule   0  manual  0   kept   32
  agg_market_monthly_means           union   42  rule   0  manual 12   kept  266
  agg_market_monthly_full_moments    union   98  rule  26  manual 57   kept  871
  panel_stock_daily_engineered       union   32  rule   0  manual  0   kept  137
  panel_stock_monthly_engineered     union   15  rule   0  manual 12   kept  158

PASS 2 - enforce identical stock-level base factors
  DAILY    aggregate  135  panel  137  shared  134  removing 4
   

In [2]:
def stock_base(tag, moment_table=False):
    """Base factors in a table. For the aggregate full-moments tables, stock
    factors are exactly those with a _cwmean column; everything else is macro."""
    cols = pd.read_parquet(OUT_DIR / f'{tag}.parquet').columns
    if moment_table:
        return {c[:-len('_cwmean')] for c in cols if c.endswith('_cwmean')}
    meta = set(META[tag] + BINARIES)
    return {c for c in cols if c not in meta}

print('=' * 90)
print('CHECK - do the panel and aggregate hold the same stock-level factors?')
print('=' * 90)
print('The union equalises the DROP lists. It cannot remove a panel column the')
print('aggregate never had, so any asymmetry here is a starting-set difference,')
print('not a union failure.\n')

for freq, agg_tag, pan_tag in [
    ('DAILY',   'agg_market_daily_full_moments',   'panel_stock_daily_engineered'),
    ('MONTHLY', 'agg_market_monthly_full_moments', 'panel_stock_monthly_engineered'),
]:
    a = stock_base(agg_tag, moment_table=True)
    p = stock_base(pan_tag)
    print(f'  {freq:<8} aggregate {len(a):>4}   panel {len(p):>4}   shared {len(a & p):>4}')
    if p - a:
        print(f'    panel only ({len(p - a)}): {sorted(p - a)}')
    if a - p:
        print(f'    aggregate only ({len(a - p)}): {sorted(a - p)}')

print('\n' + '=' * 90)
print('CHECK - the means table and the full-moments _cwmean column must agree')
print('=' * 90)
for m_tag, f_tag in [('agg_market_daily_means',   'agg_market_daily_full_moments'),
                     ('agg_market_monthly_means', 'agg_market_monthly_full_moments')]:
    m = set(pd.read_parquet(OUT_DIR / f'{m_tag}.parquet').columns)
    f = set(pd.read_parquet(OUT_DIR / f'{f_tag}.parquet').columns)
    stock = {c[:-len('_cwmean')] for c in f if c.endswith('_cwmean')}
    meta = set(META[m_tag] + BINARIES)
    macro = {c for c in f if c in m and c not in meta and c not in stock}
    expected = stock | macro | (meta & m)
    diff = (m - expected) | (expected - m)
    print(f'  {m_tag:<34} {"OK" if not diff else f"MISMATCH: {sorted(diff)}"}')

CHECK - do the panel and aggregate hold the same stock-level factors?
The union equalises the DROP lists. It cannot remove a panel column the
aggregate never had, so any asymmetry here is a starting-set difference,
not a union failure.

  DAILY    aggregate  134   panel  134   shared  134
  MONTHLY  aggregate  154   panel  154   shared  154

CHECK - the means table and the full-moments _cwmean column must agree
  agg_market_daily_means             OK
  agg_market_monthly_means           OK


In [3]:
print('=' * 90)
print('DATE COVERAGE  (informs the START_DATE trim in notebook 02)')
print('=' * 90)
for tag in SOURCES:
    d = pd.read_parquet(OUT_DIR / f'{tag}.parquet', columns=['date'])['date']
    print(f'  {tag:<34} {d.min().date()} .. {d.max().date()}   '
          f'{d.nunique():>6,} dates')

agg = set(pd.read_parquet(OUT_DIR / 'agg_market_daily_means.parquet',
                          columns=['date'])['date'])
pan = set(pd.read_parquet(OUT_DIR / 'panel_stock_daily_engineered.parquet',
                          columns=['date'])['date'])
extra = sorted(pan - agg)
print(f'\n  dates in panel but not aggregate: {len(extra)}')
if extra:
    print(f'    range {extra[0].date()} .. {extra[-1].date()}')
    print(f'    after 2007-08-01: {sum(d >= pd.Timestamp("2007-08-01") for d in extra)}')

DATE COVERAGE  (informs the START_DATE trim in notebook 02)
  agg_market_daily_means             2004-03-16 .. 2024-12-30    5,234 dates
  agg_market_daily_full_moments      2004-03-16 .. 2024-12-30    5,234 dates
  weekly_raw                         2004-01-02 .. 2024-12-30    3,107 dates
  agg_market_monthly_means           2004-04-30 .. 2024-11-30      248 dates
  agg_market_monthly_full_moments    2004-04-30 .. 2024-11-30      248 dates
  panel_stock_daily_engineered       2004-01-02 .. 2024-12-31    5,285 dates
  panel_stock_monthly_engineered     2004-01-31 .. 2024-12-31      252 dates

  dates in panel but not aggregate: 51
    range 2004-01-02 .. 2024-12-31
    after 2007-08-01: 1
